# Configuration

## Install necessary libraries from python

In [1]:
import os
import pickle
import subprocess
import shutil

In [2]:
%%capture
!pip install transformers datasets torch
!pip install git+https://github.com/huggingface/accelerate
# !pip install dgl==2.0.0 -f https://data.dgl.ai/wheels/cu121/repo.html
!sudo apt-get -q install graphviz graphviz-dev
!pip install -q pygraphviz
!pip install -q slither-analyzer==0.8.0
!pip install dgl==1.1.2
!pip install py-solc==3.2.0
!pip install networkx==2.5.1
!solc-select install 0.4.25
!solc-select use 0.4.25
!which solc && solc --version

In [3]:
# %%capture
file_path = '/kaggle/input/sc-vul-detection-dataset/SmartContractVulnerabilityDetection/sc_versions.pkl'
with open(file_path, 'rb') as f:
    sc_versions = pickle.load(f)

destination_path = '/content/ge-sc/artifacts'
for sc_version in sc_versions:

    print(sc_version)
    try:
        subprocess.run(['solc-select', 'install', sc_version])
        solc_compiler = os.path.expanduser(f'~/.solc-select/artifacts/solc-{sc_version}')
        shutil.copytree(solc_compiler, destination_path, dirs_exist_ok=True)
    except Exception as e:
        print(sc_version)
        print(e)

0.4.20
0.4.99
0.4.99
[Errno 2] No such file or directory: '/root/.solc-select/artifacts/solc-0.4.99'
0.4.9
0.4.22
0.4.19
0.4.2
0.4.11
0.4.10
0.4.23
0.5.5
0.5.2
0.5.7
0.5.0
0.4.8
0.4.7
0.5.8
0.4.4
0.4.15
0.4.24
0.4.12
0.4.16
0.4.13
0.4.26
0.5.9
0.5.4
0.5.6
0.4.0
0.5.3
0.4.21
0.5.1
0.4.17
0.8.0
0.4.18
0.4.25
0.4.14
0.4.6


## Import Python libraries

In [4]:
from concurrent.futures import ThreadPoolExecutor
from random import sample
import json
import multiprocessing
import traceback
import pandas as pd
import os
from pathlib import Path
import networkx as nx
import matplotlib.pyplot as plt
import dgl
import shutil
import random
import re
import logging
from copy import deepcopy
from os.path import join
from scipy.integrate._ivp.radau import C
from slither.slither import Slither
from collections import defaultdict
from networkx.algorithms import cluster
from slither.core.cfg.node import Node, NodeType
from slither.printers.call import call_graph
from slither.printers.abstract_printer import AbstractPrinter
from slither.core.declarations.solidity_variables import SolidityFunction
from slither.core.declarations.function import Function
from slither.core.variables.variable import Variable
import glob
from multiprocessing import Pool as ThreadPool
from functools import partial
import torch.nn as nn
import torch
from torch import Tensor
import torch.nn.functional as F
from dgl.nn import GraphConv
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
import pandas as pd
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
import pickle
import random
import numpy as np
from tqdm import tqdm
from shutil import copy
from re import L
from typing import Pattern

DGL backend not selected or invalid.  Assuming PyTorch for now.


Setting the default backend to "pytorch". You can change it in the ~/.dgl/config.json file or export the DGLBACKEND environment variable.  Valid options are: pytorch, mxnet, tensorflow (all lowercase)


# Solidity code to Graph

## Generate call-graph

In [5]:
logger = logging.getLogger("Slither-simil")

def seed_everything(seed: int):
    # Hàm đặt seed cố định cho tất cả các thư viện để kết quả có thể tái tạo lại được
    import random, os
    import numpy as np
    import torch
    
    random.seed(seed)  # Cố định seed cho thư viện random
    # os.environ['PYTHONHASHSEED'] = str(seed)  # Cố định hash seed của Python
    np.random.seed(seed)  # Cố định seed cho NumPy
    torch.manual_seed(seed)  # Cố định seed cho PyTorch CPU
    torch.cuda.manual_seed(seed)  # Cố định seed cho PyTorch GPU
    torch.backends.cudnn.deterministic = True  # Đảm bảo tính nhất quán của thuật toán cuDNN
    torch.backends.cudnn.benchmark = True  # Tối ưu hóa tốc độ tính toán

# get sol version
def get_solc_version(source):
    pattern =  re.compile(r'\d.\d.\d+')
    with open(source, 'r') as f:
        line = f.readline()
        while line:
            if 'pragma solidity' in line:
                if len(pattern.findall(line)) > 0:
                    return pattern.findall(line)[0]
                else:
                    return '0.4.25'
            line = f.readline()
    return '0.4.25'

# Contract function node
def _function_node(contract, function, filename_input):
    node_function_source_code_start = function.source_mapping['start']
    node_function_source_code_length = function.source_mapping['length']
    node_info = {
        'node_id': f"{filename_input}_{contract.id}_{contract.name}_{function.full_name}",
        'label': f"{filename_input}_{contract.name}_{function.full_name}",
        'function_fullname': function.full_name,
        'contract_name': contract.name,
        'source_file': filename_input,
        'node_source_code_start': node_function_source_code_start,
        'node_source_code_length': node_function_source_code_length,
        'visibility': function.visibility
    }
    return node_info

# Solidity function node
def _solidity_function_node(solidity_function):
    node_info = {
        'node_id': f"[Solidity]_{solidity_function.full_name}",
        'label': f"[Solidity]_{solidity_function.full_name}",
        'function_fullname': solidity_function.full_name,
        'contract_name': None,
        'source_file': None,
        'node_source_code_start': None,
        'node_source_code_length': None,
        'visibility': 'public'
    }
    return node_info

# return node info from a node tupple
def _get_node_info(tuple_node):
    if tuple_node[0][0] == 'node_id':
        node_id = tuple_node[0][1]
    if tuple_node[1][0] == 'label':
        node_label = tuple_node[1][1]
    if tuple_node[2][0] == 'function_fullname':
        function_fullname = tuple_node[2][1]
    if tuple_node[3][0] == 'contract_name':
        contract_name = tuple_node[3][1]
    if tuple_node[4][0] == 'source_file':
        source_file = tuple_node[4][1]
    if tuple_node[5][0] == 'node_source_code_start':
        node_function_source_code_start = tuple_node[5][1]
    if tuple_node[6][0] == 'node_source_code_length':
        node_function_source_code_length = tuple_node[6][1]
    if tuple_node[7][0] == 'visibility':
        visibility = tuple_node[7][1]

    if 'fallback' in node_id:
        node_type = 'fallback_function'
    elif '[Solidity]' in node_id:
        node_type = 'fallback_function'
    else:
        node_type = 'contract_function'

    return node_id, node_label, node_type, function_fullname, contract_name, source_file, node_function_source_code_start, node_function_source_code_length, visibility

# return edge info from a contract call tuple
def _add_edge_info_to_nxgraph(contract_call, nx_graph):
    source = contract_call[0]
    source_node_id, source_label, source_type, source_function_fullname, source_contract_name, \
    source_source_file, source_node_function_source_code_start, source_node_function_source_code_length, source_visibility = _get_node_info(source)

    if source_node_id not in nx_graph.nodes():
        nx_graph.add_node(source_node_id, label=source_label, node_type=source_type,
                          node_source_code_start=source_node_function_source_code_start, node_source_code_length=source_node_function_source_code_length,
                          function_fullname=source_function_fullname,
                          function_vis=source_visibility, contract_name=source_contract_name,
                          source_file=source_source_file)

    target = contract_call[1]
    target_node_id, target_label, target_type, target_function_fullname, target_contract_name, \
    target_source_file, target_node_function_source_code_start, target_node_function_source_code_length,  target_visibility = _get_node_info(target)

    if target_node_id not in nx_graph.nodes():
        nx_graph.add_node(target_node_id, label=target_label, node_type=target_type,
                          node_source_code_start=target_node_function_source_code_start, node_source_code_length=target_node_function_source_code_length,
                          function_fullname=target_function_fullname,
                          function_vis=target_visibility, contract_name=target_contract_name,
                          source_file=target_source_file)

    edge_type = contract_call[2]
    edge_label = contract_call[3]

    nx_graph.add_edge(source_node_id, target_node_id, label=edge_label, edge_type=edge_type)

def _process_internal_call(
    contract,
    function,
    internal_call,
    contract_calls,
    solidity_functions,
    solidity_calls,
    filename_input
):
    if isinstance(internal_call, (Function)):
        contract_calls[contract].add(
            (
                tuple(_function_node(contract, function, filename_input).items()),
                tuple(_function_node(contract, internal_call, filename_input).items()),
                'internal_call',
                'internal_call'
            )
        )

    elif isinstance(internal_call, (SolidityFunction)):
        solidity_functions.add(tuple(_solidity_function_node(internal_call).items()))
        solidity_calls.add(
            (
                tuple(_function_node(contract, function, filename_input).items()),
                tuple(_solidity_function_node(internal_call).items()),
                'solidity_call',
                'solidity_call'
            )
        )

def _process_external_call(
    contract,
    function,
    external_call,
    contract_functions,
    external_calls,
    all_contracts,
    filename_input
):
    external_contract, external_function = external_call
    if not external_contract in all_contracts:
        return

    if isinstance(external_function, (Variable)):
        contract_functions[external_contract].add(tuple(
                _function_node(external_contract, external_function, filename_input).items()))

    external_calls.add(
        (
            tuple(_function_node(contract, function, filename_input).items()),
            tuple(_function_node(external_contract, external_function, filename_input).items()),
            'external_call',
            'external_call'
        )
    )

def _process_function(
    contract,
    function,
    contract_functions,
    contract_calls,
    solidity_functions,
    solidity_calls,
    external_calls,
    all_contracts,
    filename_input
):
    contract_functions[contract].add(tuple(
        _function_node(contract, function, filename_input).items())
    )
    for internal_call in function.internal_calls:
        _process_internal_call(
            contract,
            function,
            internal_call,
            contract_calls,
            solidity_functions,
            solidity_calls,
            filename_input
        )
    for external_call in function.high_level_calls:

        _process_external_call(
            contract,
            function,
            external_call,
            contract_functions,
            external_calls,
            all_contracts,
            filename_input
        )

def _process_functions(functions, filename_input, vulnerabilities_in_sc=None):
    contract_functions = defaultdict(set)  # contract -> contract functions nodes
    contract_calls = defaultdict(set)  # contract -> contract calls edges

    solidity_functions = set()  # solidity function nodes
    solidity_calls = set()  # solidity calls edges

    external_calls = set()  # external calls edges

    all_contracts = set()
    for function in functions:
        all_contracts.add(function.contract_declarer)

    for function in functions:
        _process_function(
            function.contract_declarer,
            function,
            contract_functions,
            contract_calls,
            solidity_functions,
            solidity_calls,
            external_calls,
            all_contracts,
            filename_input
        )

    all_contracts_graph = nx.MultiDiGraph()
    for contract in all_contracts:
        if len(contract_functions[contract]) > 0:
            for contract_function in contract_functions[contract]:
                node_id, node_label, node_type, function_fullname, contract_name, source_file, \
                node_function_source_code_start, node_function_source_code_length, source_visibility = _get_node_info(contract_function)

                all_contracts_graph.add_node(node_id, label=node_label, node_type=node_type,
                                  node_source_code_start=node_function_source_code_start, node_source_code_length=node_function_source_code_length,
                                  function_fullname=function_fullname, function_vis=source_visibility, contract_name=contract_name,
                                  source_file=source_file)

        if len(contract_calls[contract]) > 0:
            for contract_call in contract_calls[contract]:
                _add_edge_info_to_nxgraph(contract_call, all_contracts_graph)

    if len(external_calls) > 0:
        for external_call in external_calls:
            _add_edge_info_to_nxgraph(external_call, all_contracts_graph)

    return all_contracts_graph

## Generate control-flow graph

In [6]:
def get_node_info(node):
    node_label = "Node Type: {}\n".format(str(node.type))
    node_type = str(node.type)
    if node.expression:
        node_label += "\nEXPRESSION:\n{}\n".format(node.expression)
        node_expression = str(node.expression)
    else:
        node_expression = None
    if node.irs:
        node_label += "\nIRs:\n" + "\n".join([str(ir) for ir in node.irs])
        node_irs = "\n".join([str(ir) for ir in node.irs])
    else:
        node_irs = None

    # print(node_label)
    node_source_code_start = node.source_mapping['start']
    node_source_code_length = node.source_mapping['length']

    return node_label, node_type, node_expression, node_irs, node_source_code_start, node_source_code_length

## Merging Cfgs to Fcgs

In [7]:
def mapping_cfg_and_cg_node_labels(cfg, call_graph):
    dict_node_label_cfg_and_cg = {}

    for node, node_data in cfg.nodes(data=True):
        if node_data['node_type'] == 'FUNCTION_NAME':
            if node_data['label'] not in dict_node_label_cfg_and_cg:
                dict_node_label_cfg_and_cg[node_data['label']] = None

            dict_node_label_cfg_and_cg[node_data['label']] = {
                'cfg_node_id': node,
                'cfg_node_type': node_data['node_type']
            }

    for node, node_data in call_graph.nodes(data=True):
        if node_data['label'] in dict_node_label_cfg_and_cg:
            dict_node_label_cfg_and_cg[node_data['label']]['call_graph_node_id'] = node
            dict_node_label_cfg_and_cg[node_data['label']]['call_graph_node_type'] = node_data['node_type'].upper()
        else:
            print(node_data['label'], ' is not existing.')

    temp_dict = dict(dict_node_label_cfg_and_cg)
    for key, value in temp_dict.items():
        if 'call_graph_node_id' not in value or 'call_graph_node_type' not in value:
            dict_node_label_cfg_and_cg.pop(key, None)

    return dict_node_label_cfg_and_cg

def add_new_cfg_edges_from_call_graph(cfg, dict_node_label, call_graph):
    list_new_edges_cfg = []
    for source, target, edge_data in call_graph.edges(data=True):
        source_cfg = None
        target_cfg = None
        edge_data_cfg = edge_data
        for value in dict_node_label.values():
            if value['call_graph_node_id'] == source:
                source_cfg = value['cfg_node_id']

            if value['call_graph_node_id'] == target:
                target_cfg = value['cfg_node_id']

        if source_cfg is not None and target_cfg is not None:
            list_new_edges_cfg.append((source_cfg, target_cfg, edge_data_cfg))

    cfg.add_edges_from(list_new_edges_cfg)

    return cfg

def update_cfg_node_types_by_call_graph_node_types(cfg, dict_node_label):
    for value in dict_node_label.values():
        cfg_node_id = value['cfg_node_id']
        cfg.nodes[cfg_node_id]['node_type'] = value['call_graph_node_type']

## Call Graph Only

In [8]:
def clean_name_functin(func_name):
    return func_name.replace("_", ".")
def get_call_graph(contract_path):
    sc_version = '0.4.24'
    pattern =  re.compile(r'\d.\d.\d+')
    with open(contract_path, 'r') as f:
        line = f.readline()
        while line:
            if 'pragma solidity' in line:
                if len(pattern.findall(line)) > 0:
                    sc_version = pattern.findall(line)[0]
                    break
                else:
                    sc_version = '0.4.24'
            line = f.readline()

    print(sc_version)
    solc_compiler = f'/content/ge-sc/artifacts/solc-{sc_version}'
    if not os.path.exists(solc_compiler):
        solc_compiler = f'/content/ge-sc/artifacts/solc-0.4.24'
    try:
        slither = Slither(contract_path, solc=solc_compiler)
    except Exception as e:
        print("Error compiling:", e)
        print("So change to default version 0.4.24")
        solc_compiler = f'/content/ge-sc/artifacts/solc-0.4.24'
        try:
            slither = Slither(contract_path, solc=solc_compiler)
            print("Fixed sucessfully!")
        except Exception as e2:
            print("Still error so give up!")
            return
        pass
    # Extract call graph
    all_functionss = [compilation_unit.functions for compilation_unit in slither.compilation_units]
    all_modifierss = [compilation_unit.modifiers for compilation_unit in slither.compilation_units]
    all_functions = [item for sublist in all_functionss for item in sublist]
    all_modifiers = [item for sublist in all_modifierss for item in sublist]
    all_functions = all_functions + all_modifiers
    all_functions_as_dict = {function.canonical_name: function for function in all_functions}

    file_name_sc = contract_path.split('/')[-1:][0]
    all_contracts_call_graph = _process_functions(all_functions_as_dict.values(), file_name_sc)

    # mapping_code = {}
    # # print("graph", all_functions[0].nodes[0].source_mapping)
    # for function in all_functions:
    #     print(function)
    #     # Get function source code
    #     function_code = ""
    #     if function.nodes:
    #         # Get the source mapping for the first node to find the file
    #         src_mapping = function.nodes[0].source_mapping
    #         with open(contract_path, 'r') as f:
    #             lines = f.readlines()
    #             # Slither line numbers are 1-based
    #             start_line = src_mapping['lines'][0] - 1
    #             end_line = src_mapping['lines'][-1]
    #             function_code = "".join(lines[start_line:end_line])
    #         mapping_code[clean_name_functin(function.canonical_name)] = function_code
    return all_contracts_call_graph

# Sol Files to Fcg Files

In [9]:
from transformers import RobertaTokenizer, RobertaModel
import torch

seed_everything(42)

# Load the tokenizer and model
tokenizer = RobertaTokenizer.from_pretrained("Quangnguyen711/codebert-syntax-solidity-re-entrancy")
model = RobertaModel.from_pretrained("Quangnguyen711/codebert-syntax-solidity-re-entrancy")

def get_embeddings(text):
    # Tokenize the input text
    inputs = tokenizer(text, return_tensors='pt', truncation=True, padding=True)

    # Get the model output
    with torch.no_grad():
        outputs = model(**inputs)

    # Get the embeddings (we use the embeddings of the [CLS] token)
    embeddings = outputs.last_hidden_state[:, 0, :].squeeze().cpu().numpy()

    return embeddings

def extract_function_code(source_file, start, length):
    with open(source_file, 'r') as f:
        source_code = f.read()

    # Extract the function's code
    function_code = source_code[start:start + length]

    return function_code

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/958 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/700 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

In [10]:
!mkdir /kaggle/working/SmartContractVulnerabilityDetection

In [11]:
os.mkdir("/kaggle/working/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset")
os.mkdir("/kaggle/working/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset")
os.mkdir("/kaggle/working/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset")

os.mkdir("/kaggle/working/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Train")
os.mkdir("/kaggle/working/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Test")
os.mkdir("/kaggle/working/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Train")
os.mkdir("/kaggle/working/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Test")

os.mkdir("/kaggle/working/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Train/Vulnerable_Fcg")
os.mkdir("/kaggle/working/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Train/NonVulnerable_Fcg")
os.mkdir("/kaggle/working/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Test/Vulnerable_Fcg")
os.mkdir("/kaggle/working/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Test/NonVulnerable_Fcg")

os.mkdir("/kaggle/working/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Train/Vulnerable_Fcg")
os.mkdir("/kaggle/working/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Train/NonVulnerable_Fcg")
os.mkdir("/kaggle/working/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Test/Vulnerable_Fcg")
os.mkdir("/kaggle/working/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Test/NonVulnerable_Fcg")

In [12]:
def clean_function(function_list):
    """Clean function names by removing address prefix and replacing underscores."""
    cleaned_functions = [re.sub(r"0x[0-9a-z]+\.sol_\d+_", "", func).replace("_", ".") for func in function_list]
    return cleaned_functions

def processSolFile(solFileSrc, fcgFileDst):
    """Process a Solidity file to generate a DGL graph with contract_index."""
    try:
        fcgFileDst = Path(fcgFileDst)
        fcg_file = fcgFileDst / f'{Path(solFileSrc).stem}.fcg'
        print(f"Processing {solFileSrc}")
        # Generate call graph and code mappings
        G = get_call_graph(solFileSrc)
        G = nx.DiGraph(G)
        
        if len(G.nodes()) == 0:
            print(f"Compiler failed on: {solFileSrc}")
            return None
        
        # Compute graph metrics and contract indices
        mappings, mappingsH, contract_indices, mapping_code, node_mapping = {}, {}, {}, {}, {}
        katz = nx.katz_centrality(G)
        closeness = nx.closeness_centrality(G)
        clustering = nx.clustering(G)
        
        # Map contracts to indices
        contract_names = [data['contract_name'] for node, data in G.nodes(data=True)]
        unique_contracts = sorted(set(contract_names))
        contract_to_idx = {name: idx for idx, name in enumerate(unique_contracts)}
        
        for idx, (node, data) in enumerate(G.nodes(data=True)):
            # print(data)
            mappings[node] = [
                G.in_degree(node),
                G.out_degree(node),
                katz[node],
                closeness[node],
                clustering[node]
            ]
            code_start = data['node_source_code_start']
            code_length = data['node_source_code_length']
            function_name = data['label'].split(".sol_")[1]
            function_name = function_name.replace("_", ".") 
            function_code = extract_function_code(solFileSrc, code_start, code_length)
            mapping_code[function_name] = function_code
            node_mapping[function_name] = idx
            mappingsH[node] = get_embeddings(extract_function_code(solFileSrc, code_start, code_length))
            contract_indices[node] = contract_to_idx[data['contract_name']]
            
        # Set node attributes
        nx.set_node_attributes(G, mappings, 'features')
        nx.set_node_attributes(G, mappingsH, 'featuresH')
        nx.set_node_attributes(G, contract_indices, 'contract_index')
        
        # Convert to DGL graph
        cg = nx.convert_node_labels_to_integers(G)
        dg = dgl.from_networkx(cg, node_attrs=['features', 'featuresH', 'contract_index'])
        
        # Save DGL graph if it doesn't exist
        if not os.path.exists(fcg_file):
            dgl.data.utils.save_graphs(str(fcg_file), [dg])
            print(f"Saved FCG: {fcg_file}")

        # Save mappings to JSON
        mapping_node_code = {
            "node": node_mapping,
            "code": mapping_code
        }

        # print(mapping_node_code)
        json_path = fcgFileDst / f'{Path(solFileSrc).stem}_mapping.json'
        os.makedirs(fcgFileDst, exist_ok=True)
        with open(json_path, "w", encoding="utf-8") as f:
            json.dump(mapping_node_code, f, indent=4, ensure_ascii=False)
        print(f"Saved mapping: {json_path}")
        
        return node_mapping, mapping_code
    
    except Exception as e:
        print(f"Error processing {solFileSrc}: {str(e)}")
        return None

"""Process Solidity files in parallel and generate FCG files with mappings."""
seed_everything(42)

src = "/kaggle/input/sc-vul-detection-dataset/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset"
src2 = "/kaggle/working/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset"
types = ["ReentrancyDataset"]
common_path = ["Test/NonVulnerable", "Test/Vulnerable", "Train/NonVulnerable", "Train/Vulnerable"]
save_common = ["Test/NonVulnerable_Fcg", "Test/Vulnerable_Fcg", "Train/NonVulnerable_Fcg", "Train/Vulnerable_Fcg"]

for tp in types:
    for idx in range(len(common_path)):
        source = os.path.join(src, tp, common_path[idx])
        destination = os.path.join(src2, tp, save_common[idx])
        
        print(f"Source: {source}")
        print(f"Destination: {destination}")
        print("-" * 100)
        
        sol_files = glob.glob(os.path.join(source, "*.sol"))
        fcg_files = [f[:-4] for f in os.listdir(destination) if f.endswith('.fcg')]
        sol_stems = [os.path.basename(f)[:-4] for f in sol_files]
        
        checkpoint = set(sol_stems) - set(fcg_files)
        cp_sol_files = [f for f in sol_files if os.path.basename(f)[:-4] in checkpoint]
        
        print(f"Unprocessed files: {len(cp_sol_files)}")
        
        with ThreadPool(4) as pool:
            results = pool.map(partial(processSolFile, fcgFileDst=destination), cp_sol_files)



# src = "/kaggle/input/sc-vul-detection-dataset/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Test/NonVulnerable/0xae5a801527695745d42034eb236662927ab1f95b.sol"
# dst = "/kaggle/working/"
# processSolFile(src, dst)

Source: /kaggle/input/sc-vul-detection-dataset/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Test/NonVulnerable
Destination: /kaggle/working/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Test/NonVulnerable_Fcg
----------------------------------------------------------------------------------------------------
Unprocessed files: 74
Processing /kaggle/input/sc-vul-detection-dataset/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Test/NonVulnerable/0xae5a801527695745d42034eb236662927ab1f95b.solProcessing /kaggle/input/sc-vul-detection-dataset/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Test/NonVulnerable/10604.solProcessing /kaggle/input/sc-vul-detection-dataset/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Test/NonVulnerable/0x7d6ccc156bbaa5220b62cb0f8eb322fdac92a2d1.sol


Processing /kaggle/input/sc-vul-detection-dataset/SmartCon

/usr/local/lib/python3.10/dist-packages/dgl/backend/pytorch/tensor.py:53: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at ../torch/csrc/utils/tensor_new.cpp:278.)
  return th.as_tensor(data, dtype=dtype)


Saved FCG: /kaggle/working/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Test/NonVulnerable_Fcg/0x7d6ccc156bbaa5220b62cb0f8eb322fdac92a2d1.fcg
Saved mapping: /kaggle/working/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Test/NonVulnerable_Fcg/0x7d6ccc156bbaa5220b62cb0f8eb322fdac92a2d1_mapping.json
Processing /kaggle/input/sc-vul-detection-dataset/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Test/NonVulnerable/simple_dao_fixed.sol
0.4.24
Saved FCG: /kaggle/working/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Test/NonVulnerable_Fcg/simple_dao_fixed.fcg
Saved mapping: /kaggle/working/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Test/NonVulnerable_Fcg/simple_dao_fixed_mapping.json
Processing /kaggle/input/sc-vul-detection-dataset/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Test/NonVulnerable/0xbe

/usr/local/lib/python3.10/dist-packages/dgl/backend/pytorch/tensor.py:53: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at ../torch/csrc/utils/tensor_new.cpp:278.)
  return th.as_tensor(data, dtype=dtype)


Saved FCG: /kaggle/working/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Test/NonVulnerable_Fcg/23387.fcg
Saved mapping: /kaggle/working/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Test/NonVulnerable_Fcg/23387_mapping.json
Processing /kaggle/input/sc-vul-detection-dataset/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Test/NonVulnerable/0x4566d68ea96fc2213f2446f0dd0f482146cee96d.sol
0.4.24


...

Saved FCG: /kaggle/working/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Train/Vulnerable_Fcg/0x17f68886d00845867c154c912b4ccc506ec92fc7.fcg
Saved mapping: /kaggle/working/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Train/Vulnerable_Fcg/0x17f68886d00845867c154c912b4ccc506ec92fc7_mapping.json
Processing /kaggle/input/sc-vul-detection-dataset/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Train/Vulnerable/0xd40775e917492a9f8afd740d52770d27682be02d.sol
0.4.19


/usr/local/lib/python3.10/dist-packages/dgl/backend/pytorch/tensor.py:53: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at ../torch/csrc/utils/tensor_new.cpp:278.)
  return th.as_tensor(data, dtype=dtype)


Saved FCG: /kaggle/working/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Train/Vulnerable_Fcg/0x5992eaf2295734d4a3d22608c304403e21e007fd.fcg
Saved mapping: /kaggle/working/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Train/Vulnerable_Fcg/0x5992eaf2295734d4a3d22608c304403e21e007fd_mapping.json
Processing /kaggle/input/sc-vul-detection-dataset/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Train/Vulnerable/0xc8d2881128dbe1534495a85edf716278b892c037.sol
0.4.21
Saved FCG: /kaggle/working/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Train/Vulnerable_Fcg/0x1ed8691cea15e9573282175ffa3e23281fce85c0.fcg
Saved mapping: /kaggle/working/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Train/Vulnerable_Fcg/0x1ed8691cea15e9573282175ffa3e23281fce85c0_mapping.json
Processing /kaggle/input/sc-vul-detection-dataset/SmartContractVulnerabilityDetecti

/usr/local/lib/python3.10/dist-packages/dgl/backend/pytorch/tensor.py:53: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at ../torch/csrc/utils/tensor_new.cpp:278.)
  return th.as_tensor(data, dtype=dtype)


Saved FCG: /kaggle/working/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Train/Vulnerable_Fcg/0x872098e7e008079040a03efcdf313ff1911769dc.fcg
Saved mapping: /kaggle/working/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Train/Vulnerable_Fcg/0x872098e7e008079040a03efcdf313ff1911769dc_mapping.json
Processing /kaggle/input/sc-vul-detection-dataset/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Train/Vulnerable/0x15d92e219cfe22b7515dd6d1cf5a6a65a4e2acf1.sol
0.4.21
Saved FCG: /kaggle/working/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Train/Vulnerable_Fcg/0x40b10014a17e997e8e55594cbfb4f085c5ec815b.fcg
Saved mapping: /kaggle/working/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Train/Vulnerable_Fcg/0x40b10014a17e997e8e55594cbfb4f085c5ec815b_mapping.json
Processing /kaggle/input/sc-vul-detection-dataset/SmartContractVulnerabilityDetecti

In [13]:
# with open(src, "r") as f:
#     print(f.read())

In [14]:
# # Load the tokenizer and model
# tokenizer = RobertaTokenizer.from_pretrained("Quangnguyen711/codebert-syntax-solidity-time-dep")
# model = RobertaModel.from_pretrained("Quangnguyen711/codebert-syntax-solidity-time-dep")

In [15]:
src = "/kaggle/input/sc-vul-detection-dataset/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset"
src2 = "/kaggle/working/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset"
types = ["TimestampDependencyDataset"]
common_path = ["Test/NonVulnerable", "Test/Vulnerable", "Train/NonVulnerable", "Train/Vulnerable"]
save_common = ["Test/NonVulnerable_Fcg", "Test/Vulnerable_Fcg", "Train/NonVulnerable_Fcg", "Train/Vulnerable_Fcg"]

for tp in types:
    for idx in range(len(common_path)):
        source = os.path.join(src, tp, common_path[idx])
        destination = os.path.join(src2, tp, save_common[idx])
        
        print(f"Source: {source}")
        print(f"Destination: {destination}")
        print("-" * 100)
        
        sol_files = glob.glob(os.path.join(source, "*.sol"))
        fcg_files = [f[:-4] for f in os.listdir(destination) if f.endswith('.fcg')]
        sol_stems = [os.path.basename(f)[:-4] for f in sol_files]
        
        checkpoint = set(sol_stems) - set(fcg_files)
        cp_sol_files = [f for f in sol_files if os.path.basename(f)[:-4] in checkpoint]
        
        print(f"Unprocessed files: {len(cp_sol_files)}")
        
        with ThreadPool(4) as pool:
            results = pool.map(partial(processSolFile, fcgFileDst=destination), cp_sol_files)

Source: /kaggle/input/sc-vul-detection-dataset/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Test/NonVulnerable
Destination: /kaggle/working/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Test/NonVulnerable_Fcg
----------------------------------------------------------------------------------------------------
Unprocessed files: 93
Processing /kaggle/input/sc-vul-detection-dataset/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Test/NonVulnerable/0x40f4d25fb85ea9c4fc4a82e8234664108b20e2cb.solProcessing /kaggle/input/sc-vul-detection-dataset/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Test/NonVulnerable/0xf487e54a41660ef17374f6ebf8340c6ef3163f30.solProcessing /kaggle/input/sc-vul-detection-dataset/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Test/NonVulnerable/0x0821126d74d1b33b5cfdedb

/usr/local/lib/python3.10/dist-packages/dgl/backend/pytorch/tensor.py:53: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at ../torch/csrc/utils/tensor_new.cpp:278.)
  return th.as_tensor(data, dtype=dtype)


Saved FCG: /kaggle/working/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Test/NonVulnerable_Fcg/0xf487e54a41660ef17374f6ebf8340c6ef3163f30.fcg
Saved mapping: /kaggle/working/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Test/NonVulnerable_Fcg/0xf487e54a41660ef17374f6ebf8340c6ef3163f30_mapping.json
Processing /kaggle/input/sc-vul-detection-dataset/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Test/NonVulnerable/0x8ae4bf2c33a8e667de34b54938b0ccd03eb8cc06.sol
0.4.24


/usr/local/lib/python3.10/dist-packages/dgl/backend/pytorch/tensor.py:53: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at ../torch/csrc/utils/tensor_new.cpp:278.)
  return th.as_tensor(data, dtype=dtype)


Saved FCG: /kaggle/working/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Test/NonVulnerable_Fcg/0x0821126d74d1b33b5cfdedbea66663868a15800c.fcg
Saved mapping: /kaggle/working/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Test/NonVulnerable_Fcg/0x0821126d74d1b33b5cfdedbea66663868a15800c_mapping.json
Processing /kaggle/input/sc-vul-detection-dataset/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Test/NonVulnerable/0x0af44e2784637218dd1d32a322d44e603a8f0c6a.sol
0.4.24
Saved FCG: /kaggle/working/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Test/NonVulnerable_Fcg/0x0af44e2784637218dd1d32a322d44e603a8f0c6a.fcg
Saved mapping: /kaggle/working/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Test/NonVulnerable_Fcg/0x0af44e2784637218dd1d32a322d44e603a8f0c6a_mapping.json
Processing /kaggle/input/sc-

...

Saved FCG: /kaggle/working/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Train/Vulnerable_Fcg/0x0f1cf136dd546a6957f0c2d3b4312685770378c4.fcg
Saved mapping: /kaggle/working/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Train/Vulnerable_Fcg/0x0f1cf136dd546a6957f0c2d3b4312685770378c4_mapping.json
Processing /kaggle/input/sc-vul-detection-dataset/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Train/Vulnerable/0x3668F174859271c88537d633a2Cac59de26B0641.sol
0.4.11
Saved FCG: /kaggle/working/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Train/Vulnerable_Fcg/0x3668F174859271c88537d633a2Cac59de26B0641.fcg
Saved mapping: /kaggle/working/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Train/Vulnerable_Fcg/0x3668F174859271c88537d633a2Cac59de26B0641_mapping.json
Processing /kaggle/input/sc-vul-detect

/usr/local/lib/python3.10/dist-packages/dgl/backend/pytorch/tensor.py:53: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at ../torch/csrc/utils/tensor_new.cpp:278.)
  return th.as_tensor(data, dtype=dtype)


Saved FCG: /kaggle/working/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Train/Vulnerable_Fcg/0xbf64a825e602a4f1c31480a470e99e1d896c88a7.fcg
Saved mapping: /kaggle/working/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Train/Vulnerable_Fcg/0xbf64a825e602a4f1c31480a470e99e1d896c88a7_mapping.json
Processing /kaggle/input/sc-vul-detection-dataset/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Train/Vulnerable/0x6afd9628ef194f2365f6de9c8cee8938dfd936ed.sol
0.4.11


/usr/local/lib/python3.10/dist-packages/dgl/backend/pytorch/tensor.py:53: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at ../torch/csrc/utils/tensor_new.cpp:278.)
  return th.as_tensor(data, dtype=dtype)


Saved FCG: /kaggle/working/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Train/Vulnerable_Fcg/0xad4c4ff144e42c73b6333b75af3cee5af901c10e.fcg
Saved mapping: /kaggle/working/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Train/Vulnerable_Fcg/0xad4c4ff144e42c73b6333b75af3cee5af901c10e_mapping.json
Processing /kaggle/input/sc-vul-detection-dataset/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Train/Vulnerable/0xf6f2e7e8b934e14f811e133fafeed1b991e9f288.sol
0.4.16
Saved FCG: /kaggle/working/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Train/Vulnerable_Fcg/0x9497043f4CD9450867479f3Fd873d80d9321094C.fcg
Saved mapping: /kaggle/working/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/TimestampDependencyDataset/Train/Vulnerable_Fcg/0x9497043f4CD9450867479f3Fd873d80d9321094C_mapping.json
Processing /kaggle/input/sc-vul-detect